# Lab 4.2 &mdash; Tool Descriptions Are Instructions

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Write an eval set for tool selection &mdash; including the cases that are genuinely ambiguous
- Build the metric: accuracy, and a confusion table that says <em>which</em> tool it wrongly picked
- Enforce the experimental control &mdash; only the description text may differ
- Run it against the live model at three description qualities &mdash; and find out whether there is a delta at all

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **This is the measured lab.** The harness is graded offline; the number comes from
> your own run against the sandbox model. You will reuse this harness on Day 3.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 4 labs -- the same payment exceptions as Day 1,
# now reached through tools the agent chooses, and then through tools it did not write.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# The tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

In [ ]:
# ------------------------------------------------- the toolkit these labs route between
# Four tools over the same ledger. Two read, one explains, one moves money -- which is the
# distinction that matters once an agent is choosing between them on its own.

def search_payments(counterparty: str = "", status: str = "") -> str:
    """Return the ledger records whose counterparty or status matches a query."""
    hits = [{"ref": r, **v} for r, v in LEDGER.items()
            if (not counterparty or v["counterparty"] == counterparty)
            and (not status or v["status"] == status)]
    return json.dumps(hits)


def release_payment(ref: str) -> str:
    """Release one held payment so that it settles."""
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, "released": True, "was": record["status"]})


TOOLKIT_FNS = {"lookup_payment": lookup_payment, "search_payments": search_payments,
               "policy_for": policy_for, "release_payment": release_payment}
print("toolkit:", ", ".join(TOOLKIT_FNS))

## Concept

The claim to test: **holding the model and the tools fixed, and changing only the description text,
moves selection accuracy.**

That is an experiment, so it needs the parts of one:

- an **eval set** &mdash; asks with a known correct tool, including ambiguous ones
- a **metric** &mdash; accuracy, plus a confusion table, because *which* wrong tool it chose tells you
  which two descriptions overlap
- a **control** &mdash; three toolsets that are identical except for the description string

The third is the one people skip, and skipping it is how you end up measuring a schema change you
forgot you made.

## Section 1 &mdash; The eval set and the three description qualities

Twelve asks. Four of them are deliberately awkward: no reference number, or a verb that could
belong to two tools. Those are the cases a boundary sentence exists for.

In [ ]:
EVAL_SET = [
    ("What is the status of PMT-1002?",                        "lookup_payment"),
    ("Show me the record for PMT-1005.",                       "lookup_payment"),
    ("Why did PMT-1004 fail? Give me its reason code.",        "lookup_payment"),
    ("I have the reference PMT-1001 -- pull it up.",           "lookup_payment"),
    ("Which payments involve NORTHWIND?",                      "search_payments"),
    ("List everything currently held for ZENITH.",             "search_payments"),
    ("Find the failed ones from ACME-EU.",                     "search_payments"),
    ("Are there any other payments like this one?",            "search_payments"),
    ("What should we do about a LIMIT_BREACH?",                "policy_for"),
    ("What is the operating rule for SANCTIONS_REVIEW?",       "policy_for"),
    ("An INVALID_IBAN came back. What does the runbook say?",  "policy_for"),
    ("Treasury approved it -- release PMT-1003.",              "release_payment"),
]

# Level 1: what the tool does. Fixes malformed arguments -- the model now knows the shape.
BASE = {
    "lookup_payment":  "Return the ledger record for one payment reference such as PMT-1002.",
    "search_payments": "Return the ledger records whose counterparty or status matches a query.",
    "policy_for":      "Return the operating policy for one failure reason code such as LIMIT_BREACH.",
    "release_payment": "Release one held payment so that it settles.",
}

# Level 2 adds one sentence each: where this tool stops, and what to reach for instead.
BOUNDARY = {
    "lookup_payment":  " Use when you already have the reference. Not for searching or listing --"
                       " use search_payments when you do not have a reference.",
    "search_payments": " Use when you must find which payments match. Not for one known reference --"
                       " use lookup_payment for that.",
    "policy_for":      " Use once you know why a payment failed. Not for reading the payment itself --"
                       " use lookup_payment for that.",
    "release_payment": " Use only after a human has approved this specific release."
                       " Not for reading, searching or explaining. This one moves money.",
}

TOOLSETS = {
    "L0 label only":  {n: f"{n.replace('_', ' ').capitalize()}." for n in BASE},
    "L1 what it does": dict(BASE),
    "L2 with boundary": {n: BASE[n] + BOUNDARY[n] for n in BASE},
}

print(f"{len(EVAL_SET)} asks, {len(TOOLSETS)} description qualities, {len(BASE)} tools")

In [ ]:
# --- Self-check: Section 1
check("the eval set is big enough to say anything",
      lambda: len(EVAL_SET) >= 12)
check("every tool in the toolkit is the right answer at least once",
      lambda: {exp for _, exp in EVAL_SET} == set(BASE))
check("no ask is expected to route to a tool that does not exist",
      lambda: all(exp in TOOLKIT_FNS for _, exp in EVAL_SET))
check("there are ambiguous asks -- ones with no reference number in them",
      lambda: sum(1 for a, _ in EVAL_SET if "PMT-" not in a) >= 4,
      "an eval set of only easy cases measures nothing")
check("all three toolsets exist and are the same size",
      lambda: len({len(t) for t in TOOLSETS.values()}) == 1)

## Section 2 &mdash; The metric

Accuracy alone tells you *that* it went wrong. The confusion table tells you *which two descriptions
overlap*, which is the thing you can actually go and edit.

A `selections` value is just `{ask: chosen_tool}` &mdash; whatever produced it.

In [ ]:
def accuracy(selections: dict, evalset=None) -> float:
    """Fraction of asks routed to the expected tool. An ask with no selection counts as wrong."""
    evalset = EVAL_SET if evalset is None else evalset
    hits = sum(1 for ask, expected in evalset if BLANK)   # TODO: was this ask routed correctly?
    return hits / len(evalset)


def confusion(selections: dict, evalset=None) -> dict:
    """{(expected, chosen): count} for the MISSES only -- the pairs whose descriptions overlap."""
    evalset = EVAL_SET if evalset is None else evalset
    out = {}
    for ask, expected in evalset:
        chosen = selections.get(ask)
        if chosen != expected:
            key = BLANK                                   # TODO: which pair confused it?
            out[key] = out.get(key, 0) + 1
    return out

In [ ]:
# --- Self-check: Section 2
# Fixtures: hand-written {ask: chosen} maps with known answers, so the METRIC is graded
# independently of anything that produced a selection. This is a unit test, not a measurement.
_perfect = {ask: exp for ask, exp in EVAL_SET}
_all_lookup = {ask: "lookup_payment" for ask, _ in EVAL_SET}
_one_miss = dict(_perfect); _one_miss["Which payments involve NORTHWIND?"] = "lookup_payment"

check("a perfect run scores 1.0", lambda: accuracy(_perfect) == 1.0)
check("a run that always picks lookup_payment scores 4/12",
      lambda: abs(accuracy(_all_lookup) - 4 / 12) < 1e-9)
check("one miss out of twelve", lambda: abs(accuracy(_one_miss) - 11 / 12) < 1e-9)
check("a missing selection counts as wrong, not as skipped",
      lambda: accuracy({}) == 0.0,
      "a model that returned nothing did not get the answer right")
check("a perfect run has an empty confusion table", lambda: confusion(_perfect) == {})
check("the confusion table names the pair, expected first",
      lambda: confusion(_one_miss) == {("search_payments", "lookup_payment"): 1})
check("confusion counts repeats of the same pair",
      lambda: confusion(_all_lookup)[("search_payments", "lookup_payment")] == 4)
check("the confusion table only records misses",
      lambda: sum(confusion(_all_lookup).values()) == 8)

## Section 3 &mdash; The control

The whole claim is *&ldquo;description text alone&rdquo;*. That is only true if nothing else differs.
Names must match, and so must the parameter schemas &mdash; otherwise you have quietly run a
different experiment and the number you report is worthless.

In [ ]:
import inspect

def tool_descriptor_params(fn) -> dict:
    """The parameter schema for one function -- the part that must NOT vary between toolsets."""
    props, required = {}, []
    for pname, p in inspect.signature(fn).parameters.items():
        props[pname] = {"type": "string"}
        if p.default is inspect.Parameter.empty:
            required.append(pname)
    return {"type": "object", "properties": props, "required": required}


def descriptors_for(toolset: dict) -> list:
    """The descriptor list a model would receive for one toolset."""
    return [{"name": n,
             "description": toolset[n],
             "parameters": tool_descriptor_params(TOOLKIT_FNS[n])}
            for n in sorted(toolset)]


def control_holds(toolsets: dict) -> bool:
    """True only if the toolsets differ in description text and in nothing else."""
    runs = [descriptors_for(t) for t in toolsets.values()]
    def shape(run):
        # TODO: everything about a run EXCEPT the description text
        return BLANK
    return len({json.dumps(shape(r), sort_keys=True) for r in runs}) == 1

In [ ]:
# --- Self-check: Section 3
def _short_toolset():
    """A fourth toolset that quietly drops a tool -- a different experiment, not a rerun."""
    return {**TOOLSETS,
            "rogue": {n: v for n, v in TOOLSETS["L1 what it does"].items()
                      if n != "release_payment"}}

def _wider(ref: str, mode: str) -> str:
    """The same tool with one more required argument."""
    return ""

check("the control holds for the three toolsets as written",
      lambda: control_holds(TOOLSETS) is True)
check("the control is not vacuous -- the descriptions really do differ",
      lambda: len({json.dumps([d["description"] for d in descriptors_for(t)])
                   for t in TOOLSETS.values()}) == 3)
check("a toolset that drops a tool breaks the control",
      lambda: control_holds(_short_toolset()) is False,
      "fewer tools in view is a change to the experiment, not a rerun of it")
check("the control is sensitive to a schema change too, not just to names",
      lambda: tool_descriptor_params(_wider)
              != tool_descriptor_params(TOOLKIT_FNS["lookup_payment"]))

## Section 4 &mdash; Measure it

Everything above is offline and deterministic. This is the part that produces a number, and it
needs the model, because **the model is the thing under test**.

The cell below asks the sandbox model to choose one tool per ask, three times over &mdash; once per
description quality &mdash; and feeds the results through *your* harness. If the model is not
reachable it says so and the lab still scores.

Do not expect the three rows to fan out neatly. Read the next section **after** you run it.

In [ ]:
SELECT_SYSTEM = ("You route a user's request to exactly one tool. "
                 "Reply with the tool name alone -- no punctuation, no explanation.")

def choose_with_model(request: str, toolset: dict) -> str:
    """Ask the model to pick one tool. Returns a bare tool name, or '' if it did not answer usably."""
    listing = "\n".join(f"- {d['name']}: {d['description']}" for d in descriptors_for(toolset))
    reply = ask_model(f"Tools available:\n{listing}\n\nRequest: {request}\n\nTool name:")
    word = (reply or "").strip().strip("`.\"' ").split()[0] if (reply or "").strip() else ""
    return word if word in toolset else ""


def ask_model(prompt: str) -> str:
    return ask(prompt, system=SELECT_SYSTEM)


def measure(toolset: dict) -> dict:
    """Run the whole eval set through the model once. Returns {ask: chosen}."""
    return {a: choose_with_model(a, toolset) for a, _ in EVAL_SET}


if llm_ready():
    def _run():
        rows = []
        for label, ts in TOOLSETS.items():
            sel = measure(ts)
            rows.append((label, accuracy(sel), confusion(sel)))
        print(f"{'description quality':22}{'accuracy':>10}   most-confused pair")
        print("-" * 72)
        for label, acc, conf in rows:
            worst = max(conf.items(), key=lambda kv: kv[1])[0] if conf else None
            pair = f"{worst[0]} -> {worst[1]}" if worst else "(none)"
            print(f"{label:22}{acc:>9.0%}   {pair}")
        return rows
    guard(_run)

### Read it

**You probably saw three identical rows, all at or near 100%.** That is the expected result on this
sandbox, and it is the most useful thing this lab has to tell you.

Here is what we measured on this model before writing the lab:

| eval set | L0 label only | L1 what it does | L2 with boundary |
|---|---|---|---|
| these twelve asks | 100% | 100% | 100% |
| the same, with the tool names obfuscated to `pmt_inq_01` etc. | 100% | 100% | 100% |
| ten deliberately ambiguous asks | 90% | 80% | 80% |
| the same ten, run again | 80% | 80% | &mdash; |

Three conclusions, and none of them is the one the lab's title leads you to expect.

**1. The model is at the ceiling, so there is nothing for the description to add.** Every ask here
names its intent plainly, and `lookup_payment` / `search_payments` are self-documenting. We removed
that second advantage by renaming the tools to `pmt_inq_01` and friends &mdash; the sort of name an
internal payments API really has &mdash; and it still scored 100%. When the task is easy enough, a
better description has no work left to do.

**2. On genuinely hard asks the difference appeared, and pointed the wrong way.** The richer
descriptions did *worse*, and the misses were all the same shape: asks that name a reference but
want the policy, pulled toward `lookup_payment` because its L1 text says &ldquo;such as PMT-1002&rdquo;.
A description can attract a call as easily as it can repel one.

**3. And then the fourth row cancels the third.** The same ten asks, run twice, moved by a whole
case. At ten asks one case is worth ten points, so a ten-point gap is one coin flip. **If one case
is worth more than the difference you are claiming, you have not measured anything.**

So: a null result, then a suggestive result, then a noise floor that swallows it. That is not a
failed lab &mdash; it is what measuring actually looks like, and it is why the harness is the part
worth keeping. What you now know that you did not know an hour ago:

- On *this* workload, with *this* model, description quality is not your accuracy problem. Anyone
  who spends next week rewriting docstrings for this tool set is optimising a ceiling.
- The place it could still matter is the ambiguous asks &mdash; and to tell a real 10-point effect
  from a coin flip there you need hundreds of cases, or dozens of repeats, not twelve asks and one run.
- Both of those are findings you can defend, and neither was available by reasoning about it.

Everything in Module 4 that is *not* measured here still holds, because it does not depend on this
result: a tool that returns instead of raising, a boundary sentence that stops a wrong call, an
allow-list on what comes back. Those are structural. This one was empirical, and the empirical
answer on this workload is &ldquo;no effect detectable&rdquo;.

Day 3 is where this gets sharp: the same harness, an eval set big enough to have a noise floor you
can quote, and repeats, so that &ldquo;better&rdquo; becomes a claim with an interval around it.

In [ ]:
score()

## Your turn

1. Replicate the third row. Write ten asks that name a payment reference but want a *different*
   tool &mdash; &ldquo;PMT-1004. What do the rules say about this kind?&rdquo; &mdash; and run all three
   qualities against them. Then run it twice more and see whether your ordering survives.
2. Add four asks that are genuinely ambiguous to a human too &mdash; the honest answer is &ldquo;ask a
   clarifying question&rdquo;. What should the expected value even be? (Day 3 has an answer: a fifth
   outcome, not a fifth tool.)
3. Write an L3 that adds one worked example to each description. Does it beat L2, and is the extra
   prompt cost on every single call worth it? Decide in advance how big a difference you would
   believe, given the size of your eval set.
4. Keep this file. On Day 3 you will extend this exact harness into the eval set that gates the
   capstone &mdash; same metric, more cases, and a cost budget alongside the accuracy.